# Impairment Robustness Evaluation

This notebook evaluates model robustness under various hardware impairments:
- Carrier Frequency Offset (CFO)
- I/Q Imbalance
- DC Offset
- Fading channels (Rayleigh, Rician)

In [ ]:
import sys
from pathlib import Path

src_path = Path("../src")
if src_path.exists():
    sys.path.insert(0, str(src_path.resolve()))

import numpy as np
import matplotlib.pyplot as plt
import torch

from robust_amc.data import load_radioml2016a, stratified_split
from robust_amc.models import create_pfcnn
from robust_amc.evaluation import (
    sweep_cfo,
    sweep_iq_imbalance,
    sweep_dc_offset,
    sweep_fading,
    plot_impairment_sweep_results,
)
from robust_amc.utils import get_device

## 1. Setup

In [ ]:
DATA_PATH = Path("../data/RML2016.10a_dict.pkl")
CHECKPOINT_PATH = Path("../checkpoints/baseline_2016/best_model.pt")

device = get_device("auto")
print(f"Using device: {device}")

In [ ]:
# Load model
if not CHECKPOINT_PATH.exists():
    print(f"Checkpoint not found at {CHECKPOINT_PATH}")
    print("Run: uv run python scripts/train_baseline.py")
else:
    model = create_pfcnn(num_classes=11)
    ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    print("Model loaded successfully!")

In [ ]:
# Load test data
if DATA_PATH.exists():
    data, labels, snrs = load_radioml2016a(DATA_PATH)
    splits = stratified_split(data, labels, snrs)
    test_data, test_labels, test_snrs = splits["test"]
    print(f"Test set: {len(test_labels)} samples")

## 2. CFO Sweep

In [ ]:
cfo_results = sweep_cfo(
    model, test_data, test_labels, test_snrs,
    cfo_range=(0, 5000, 11),
    device=device,
)

plt.figure(figsize=(8, 5))
plt.plot(cfo_results["cfo_hz"], cfo_results["accuracy"], "b-o", linewidth=2)
plt.xlabel("CFO (Hz)")
plt.ylabel("Accuracy")
plt.title("Accuracy vs Carrier Frequency Offset")
plt.grid(True, alpha=0.3)
plt.show()

## 3. I/Q Imbalance Sweep

In [ ]:
iq_results = sweep_iq_imbalance(
    model, test_data, test_labels, test_snrs,
    device=device,
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(iq_results["amplitude"]["amplitude_db"], iq_results["amplitude"]["accuracy"], "b-o")
axes[0].set_xlabel("Amplitude Imbalance (dB)")
axes[0].set_ylabel("Accuracy")
axes[0].set_title("Accuracy vs Amplitude Imbalance")
axes[0].grid(True, alpha=0.3)

axes[1].plot(iq_results["phase"]["phase_deg"], iq_results["phase"]["accuracy"], "g-o")
axes[1].set_xlabel("Phase Imbalance (degrees)")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Accuracy vs Phase Imbalance")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. DC Offset Sweep

In [ ]:
dc_results = sweep_dc_offset(
    model, test_data, test_labels, test_snrs,
    device=device,
)

plt.figure(figsize=(8, 5))
plt.plot(dc_results["dc_offset"], dc_results["accuracy"], "b-o", linewidth=2)
plt.xlabel("Relative DC Offset")
plt.ylabel("Accuracy")
plt.title("Accuracy vs DC Offset")
plt.grid(True, alpha=0.3)
plt.show()

## 5. Fading Channel Evaluation

In [ ]:
fading_results = sweep_fading(
    model, test_data, test_labels, test_snrs,
    n_realizations=3,
    device=device,
)

print(f"Rayleigh fading accuracy: {fading_results['rayleigh']['accuracy']:.2%}")
print("\nRician fading (by K-factor):")
for k, result in fading_results["rician"].items():
    print(f"  K={k}: {result['accuracy']:.2%} (+/- {result['std']:.2%})")

## 6. Summary Plot

In [ ]:
# Combine results for plotting
all_results = {
    "cfo": cfo_results,
    "iq": iq_results,
    "dc": dc_results,
    "fading": fading_results,
}

# Get baseline accuracy (no impairments)
baseline_acc = cfo_results["accuracy"][0]  # CFO=0 is baseline

fig = plot_impairment_sweep_results(all_results, baseline_acc)
plt.show()